# Zero-TVM throughput bench — Colab T4

**First: Runtime → Change runtime type → T4 GPU.**

Colab installs the CUDA driver but not the GL/Vulkan userspace Chrome's WebGPU needs, so cell 1 installs `libnvidia-gl-<driver>` (provides `libGLX_nvidia.so.0` + the real Vulkan ICD) — the step that turns `llvmpipe` into the actual T4. Method follows Chrome's official [colab-headless guide](https://developer.chrome.com/docs/web-platform/webgpu/colab-headless). If cell 1 still shows `llvmpipe`, the `libnvidia-gl` version didn't match the driver — tell me the driver from `nvidia-smi`.

In [ ]:
# 1) Node 22 + the NVIDIA Vulkan userspace matching the driver (the key piece)
import subprocess
drv = subprocess.check_output('nvidia-smi --query-gpu=driver_version --format=csv,noheader', shell=True).decode().split('.')[0].strip()
print('NVIDIA driver major:', drv, '-> installing libnvidia-gl-' + drv)
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1
!apt-get -qq update > /dev/null
!apt-get -qq install -y nodejs vulkan-tools libnvidia-gl-{drv} xvfb libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libgbm1 libasound2 libxcomposite1 libxdamage1 libxrandr2 libxkbcommon0 libxfixes3 libdrm2 libpango-1.0-0 libcairo2 libatspi2.0-0 fonts-liberation > /dev/null
# drop the mesa llvmpipe (software) Vulkan device so Chrome can only pick the T4
!rm -f /usr/share/vulkan/icd.d/lvp_icd*.json
print('--- Vulkan device (want Tesla T4, not llvmpipe): ---')
!vulkaninfo --summary 2>/dev/null | grep -E 'deviceName|driverName' || echo 'NO VULKAN DEVICE'

In [ ]:
# 2) Clone + run the bench. Aborts on its own if cell 1 still showed llvmpipe.
#    First run downloads ~2 GB of Phi-3 weights.
!pip -q install -U huggingface_hub
!rm -rf zero-tvm && git clone -q https://github.com/abgnydn/zero-tvm.git
!cd zero-tvm && npm ci --silent && BENCH_HEADLESS=new bash bench/cloud-bench.sh

## Done

Copy the printed `results.json` into your repo at `bench/results.json`, then run `npm run bench:sync -- --write` locally (no GPU) to update BENCH.md + the bench page.